# Case-Study Summary Figures and Z3 Runtime Analysis

This notebook reproduces paper-style summary figures from `evaluation/case-study`, and compares **naive vs optimized** behavior focusing on:

- total runtime distributions
- result-cardinality distributions
- runtime-vs-graph-metric correlations
- Z3 operation/solver runtime split by **empty** (`results == 0`) vs **non-empty** queries

All figures are exported as **PDF** files under `evaluation/figure/case-study-summary`.

## 1. Set Up Environment and Dependencies

In [2]:
import json
import math
import re
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from scipy.stats import mannwhitneyu
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["savefig.dpi"] = 180

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scipy available:", SCIPY_AVAILABLE)

pandas: 3.0.1
numpy: 2.4.4
scipy available: True


## 2. Define Configuration and Constants

In [3]:
ROOT = Path.cwd()
if ROOT.name != "evaluation":
    ROOT = ROOT / "evaluation"

CASE_STUDY_ROOT = ROOT / "case-study"
FIGURE_ROOT = ROOT / "figure" / "case-study-summary"

for sub in ["runtime", "cardinality", "correlation", "z3", "tables"]:
    (FIGURE_ROOT / sub).mkdir(parents=True, exist_ok=True)

DATASET_META = {
    "ldbc01": {"display": "LDBC01", "V": 184_000, "E": 768_000},
    "ldbc10": {"display": "LDBC10", "V": 30_000_000, "E": 178_000_000},
    "pokec": {"display": "Pokec", "V": 1_632_803, "E": 30_622_564},
    "telecom": {"display": "Telecom", "V": 170_000, "E": 50_000_000},
    "icij-leak": {"display": "ICIJ-Leak", "V": 1_908_485, "E": 3_193_392},
    "paradise": {"display": "ICIJ-Paradises", "V": 163_000, "E": 364_000},
}

ALGORITHMS = ["naive", "optimized"]
CONSTRAINT_VARIANTS = ["data-1", "data-2", "data-3", "data-4", "data-5", "no-data"]

SIMPLE_REGULAR = {1, 2, 4, 7, 11}      # Q1,Q2,Q4,Q7,Q11
MEDIUM_REGULAR = {5, 6, 9, 10}         # Q5,Q6,Q9,Q10
COMPLEX_REGULAR = {3, 8, 12}           # Q3,Q8,Q12

SIMPLE_DATA = {"data-1", "data-2"}
COMPLEX_DATA = {"data-3", "data-4", "data-5"}

FLOAT_RE = r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?"

print("CASE_STUDY_ROOT:", CASE_STUDY_ROOT)
print("FIGURE_ROOT:", FIGURE_ROOT)

CASE_STUDY_ROOT: /home/heyang-li/research/MillenniumDB/evaluation/case-study
FIGURE_ROOT: /home/heyang-li/research/MillenniumDB/evaluation/figure/case-study-summary


## 3. Prepare Input Data

In [4]:
@dataclass
class RunLeaf:
    dataset_key: str
    algorithm: str
    query_dir: str
    query_template: int
    variant: str
    path: Path


def regular_complexity(query_template: int) -> str:
    if query_template in SIMPLE_REGULAR:
        return "simple_regular"
    if query_template in MEDIUM_REGULAR:
        return "medium_regular"
    if query_template in COMPLEX_REGULAR:
        return "complex_regular"
    return "unknown_regular"


def data_complexity(variant: str) -> str:
    if variant in SIMPLE_DATA:
        return "simple_data"
    if variant in COMPLEX_DATA:
        return "complex_data"
    return "no_data"


def list_run_leaves(case_root: Path) -> tuple[list[RunLeaf], list[str]]:
    leaves: list[RunLeaf] = []
    warnings: list[str] = []

    for dataset_key in sorted(DATASET_META.keys()):
        dataset_root = case_root / dataset_key
        if not dataset_root.exists():
            warnings.append(f"Missing dataset directory: {dataset_root}")
            continue

        for algorithm in ALGORITHMS:
            algo_root = dataset_root / algorithm
            if not algo_root.exists():
                warnings.append(f"Missing algorithm directory: {algo_root}")
                continue

            for query_dir in sorted([p for p in algo_root.iterdir() if p.is_dir() and p.name.startswith("Q")], key=lambda p: int(p.name[1:])):
                q_idx = int(query_dir.name[1:]) + 1
                for variant in CONSTRAINT_VARIANTS:
                    leaf = query_dir / variant
                    if not leaf.exists():
                        continue
                    leaves.append(
                        RunLeaf(
                            dataset_key=dataset_key,
                            algorithm=algorithm,
                            query_dir=query_dir.name,
                            query_template=q_idx,
                            variant=variant,
                            path=leaf,
                        )
                    )

    return leaves, warnings


all_leaves, leaf_warnings = list_run_leaves(CASE_STUDY_ROOT)
print("leaf folders found:", len(all_leaves))
if leaf_warnings:
    print("warnings:")
    for w in leaf_warnings[:8]:
        print(" -", w)
    if len(leaf_warnings) > 8:
        print(f" ... and {len(leaf_warnings) - 8} more")

pd.DataFrame([{
    "dataset": x.dataset_key,
    "algorithm": x.algorithm,
    "query": x.query_dir,
    "variant": x.variant
} for x in all_leaves]).groupby(["dataset", "algorithm"]).size().rename("leaf_count").reset_index().head(20)

leaf folders found: 864


,dataset,algorithm,leaf_count
0,icij-leak,naive,72
1,icij-leak,optimized,72
2,ldbc01,naive,72
3,ldbc01,optimized,72
4,ldbc10,naive,72
5,ldbc10,optimized,72
6,paradise,naive,72
7,paradise,optimized,72
8,pokec,naive,72
9,pokec,optimized,72


## 4. Implement Core Functions

In [5]:
def parse_result_entry_cardinality(payload: str) -> int:
    """Parse one result payload and count returned tuple rows."""
    lines = [ln.strip() for ln in payload.splitlines()]
    nonempty = [ln for ln in lines if ln]
    if len(nonempty) <= 1:
        return 0
    return len(nonempty) - 1


def parse_result_json(path: Path) -> list[int]:
    if not path.exists():
        return []
    data = json.loads(path.read_text())
    if not data:
        return []
    key = next(iter(data.keys()))
    values = data[key]
    return [parse_result_entry_cardinality(v) for v in values]


def parse_memory_json(path: Path) -> list[float]:
    if not path.exists():
        return []
    data = json.loads(path.read_text())
    if not data:
        return []
    key = next(iter(data.keys()))
    values = data[key]
    out = []
    for v in values:
        try:
            out.append(float(v))
        except Exception:
            out.append(np.nan)
    return out


def parse_db_log(path: Path) -> list[dict]:
    if not path.exists():
        return []

    text = path.read_text(errors="ignore")
    chunks = re.findall(r"Query\(worker.*?(?=\nQuery\(worker|\Z)", text, flags=re.DOTALL)

    records = []
    for idx, chunk in enumerate(chunks):
        row = {
            "exec_idx": idx,
            "worker_results": np.nan,
            "parser_ms": np.nan,
            "optimizer_ms": np.nan,
            "execution_ms": np.nan,
            "z3_operation_ms": np.nan,
            "z3_solver_ms": np.nan,
            "solver_memory_mb": np.nan,
            "exploration_depth": np.nan,
        }

        m = re.search(r"Worker\s+\d+\s+results:\s*(\d+)", chunk)
        if m:
            row["worker_results"] = int(m.group(1))

        m = re.search(rf"Parser\s*:\s*({FLOAT_RE})\s*ms", chunk)
        if m:
            row["parser_ms"] = float(m.group(1))

        m = re.search(rf"Optimizer\s*:\s*({FLOAT_RE})\s*ms", chunk)
        if m:
            row["optimizer_ms"] = float(m.group(1))

        m = re.search(rf"Execution\s*:\s*({FLOAT_RE})\s*ms", chunk)
        if m:
            row["execution_ms"] = float(m.group(1))

        m = re.search(rf"z3_operation_time\s*[:：]\s*({FLOAT_RE})\s*ms", chunk)
        if m:
            row["z3_operation_ms"] = float(m.group(1))

        m = re.search(rf"z3_solver_time\s*[:：]\s*({FLOAT_RE})\s*ms", chunk)
        if m:
            row["z3_solver_ms"] = float(m.group(1))

        m = re.search(rf"solver_memory_consumption\s*[:：]\s*({FLOAT_RE})\s*MB", chunk)
        if m:
            row["solver_memory_mb"] = float(m.group(1))

        m = re.search(rf"exploration_depth\s*[:：]\s*({FLOAT_RE})", chunk)
        if m:
            row["exploration_depth"] = float(m.group(1))

        row["total_ms"] = row["parser_ms"] + row["optimizer_ms"] + row["execution_ms"]
        records.append(row)

    return records


def load_leaf_records(leaf: RunLeaf) -> list[dict]:
    db_records = parse_db_log(leaf.path / "db.log")
    memory_values = parse_memory_json(leaf.path / "memory.json")
    result_cardinalities = parse_result_json(leaf.path / "result.json")

    n = max(len(db_records), len(memory_values), len(result_cardinalities))
    rows = []
    for i in range(n):
        base = db_records[i].copy() if i < len(db_records) else {
            "exec_idx": i,
            "worker_results": np.nan,
            "parser_ms": np.nan,
            "optimizer_ms": np.nan,
            "execution_ms": np.nan,
            "z3_operation_ms": np.nan,
            "z3_solver_ms": np.nan,
            "solver_memory_mb": np.nan,
            "exploration_depth": np.nan,
            "total_ms": np.nan,
        }

        base["memory_json_total_mb"] = memory_values[i] if i < len(memory_values) else np.nan
        base["result_cardinality"] = int(result_cardinalities[i]) if i < len(result_cardinalities) else np.nan

        card = base["worker_results"]
        if pd.isna(card):
            card = base["result_cardinality"]
        base["is_empty"] = bool(card == 0) if not pd.isna(card) else np.nan

        base["dataset_key"] = leaf.dataset_key
        base["dataset"] = DATASET_META[leaf.dataset_key]["display"]
        base["algorithm"] = leaf.algorithm
        base["query_dir"] = leaf.query_dir
        base["query_template"] = leaf.query_template
        base["variant"] = leaf.variant
        base["data_complexity"] = data_complexity(leaf.variant)
        base["regular_complexity"] = regular_complexity(leaf.query_template)
        base["leaf_path"] = str(leaf.path)

        v = DATASET_META[leaf.dataset_key]["V"]
        e = DATASET_META[leaf.dataset_key]["E"]
        base["V"] = v
        base["E"] = e
        base["E_over_V"] = e / v if v else np.nan
        base["V_plus_E"] = v + e

        rows.append(base)

    return rows

In [6]:
def top_quartile_stats(series: pd.Series) -> tuple[float, float]:
    s = series.dropna()
    if s.empty:
        return np.nan, np.nan
    q = s.quantile(0.75)
    top = s[s >= q]
    return float(top.mean()), float(top.median())


def savefig_pdf(path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, format="pdf", bbox_inches="tight")
    plt.close()


def effect_size_cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    """Cliff's delta via rank-based U statistic; positive => x tends larger than y."""
    if len(x) == 0 or len(y) == 0:
        return np.nan
    if not SCIPY_AVAILABLE:
        return np.nan
    u, _ = mannwhitneyu(x, y, alternative="two-sided")
    return float((2 * u) / (len(x) * len(y)) - 1)


def safe_mwu_pvalue(x: np.ndarray, y: np.ndarray) -> float:
    if len(x) == 0 or len(y) == 0 or not SCIPY_AVAILABLE:
        return np.nan
    _, p = mannwhitneyu(x, y, alternative="two-sided")
    return float(p)

In [7]:
sample_leaf = all_leaves[0]
print("sample leaf:", sample_leaf.path)
sample_rows = load_leaf_records(sample_leaf)
print("sample rows parsed:", len(sample_rows))
pd.DataFrame(sample_rows).head(3)

sample leaf: /home/heyang-li/research/MillenniumDB/evaluation/case-study/icij-leak/naive/Q0/data-1
sample rows parsed: 100


,exec_idx,worker_results,parser_ms,optimizer_ms,execution_ms,z3_operation_ms,z3_solver_ms,solver_memory_mb,exploration_depth,total_ms,memory_json_total_mb,result_cardinality,is_empty,dataset_key,dataset,algorithm,query_dir,query_template,variant,data_complexity,regular_complexity,leaf_path,V,E,E_over_V,V_plus_E
0,0,0,0.140850,0.30043,0.03888,0.00002,0.00022,17.0,0.0,0.480160,126.597656,1,True,icij-leak,ICIJ-Leak,naive,Q0,1,data-1,simple_data,simple_regular,/home/heyang-li/research/MillenniumDB/evaluati...,1908485,3193392,1.67326,5101877
1,1,0,0.053860,0.11752,23.19810,0.28915,0.00008,33.0,1.0,23.369480,160.855469,1,True,icij-leak,ICIJ-Leak,naive,Q0,1,data-1,simple_data,simple_regular,/home/heyang-li/research/MillenniumDB/evaluati...,1908485,3193392,1.67326,5101877
2,2,0,0.043879,0.12455,20.01580,0.16305,0.00010,33.0,1.0,20.184229,160.937500,1,True,icij-leak,ICIJ-Leak,naive,Q0,1,data-1,simple_data,simple_regular,/home/heyang-li/research/MillenniumDB/evaluati...,1908485,3193392,1.67326,5101877


## 5. Execute the Main Workflow

In [8]:
all_rows: list[dict] = []
missing_files = []

for leaf in all_leaves:
    for fname in ["db.log", "memory.json", "result.json"]:
        if not (leaf.path / fname).exists():
            missing_files.append(str(leaf.path / fname))
    all_rows.extend(load_leaf_records(leaf))

exec_df = pd.DataFrame(all_rows)

# Normalize dtypes
num_cols = [
    "worker_results", "parser_ms", "optimizer_ms", "execution_ms", "total_ms",
    "z3_operation_ms", "z3_solver_ms", "solver_memory_mb", "exploration_depth",
    "memory_json_total_mb", "result_cardinality", "V", "E", "E_over_V", "V_plus_E"
]
for c in num_cols:
    exec_df[c] = pd.to_numeric(exec_df[c], errors="coerce")

exec_df["is_empty"] = exec_df["is_empty"].astype("boolean")

print("Execution rows:", len(exec_df))
print("Missing file references:", len(missing_files))
print("Datasets:", sorted(exec_df["dataset"].dropna().unique()))
print("Algorithms:", sorted(exec_df["algorithm"].dropna().unique()))

exec_df.head(3)

Execution rows: 86610
Missing file references: 0
Datasets: ['ICIJ-Leak', 'ICIJ-Paradises', 'LDBC01', 'LDBC10', 'Pokec', 'Telecom']
Algorithms: ['naive', 'optimized']


,exec_idx,worker_results,parser_ms,optimizer_ms,execution_ms,z3_operation_ms,z3_solver_ms,solver_memory_mb,exploration_depth,total_ms,memory_json_total_mb,result_cardinality,is_empty,dataset_key,dataset,algorithm,query_dir,query_template,variant,data_complexity,regular_complexity,leaf_path,V,E,E_over_V,V_plus_E
0,0,0.0,0.140850,0.30043,0.03888,0.00002,0.00022,17.0,0.0,0.480160,126.597656,1.0,True,icij-leak,ICIJ-Leak,naive,Q0,1,data-1,simple_data,simple_regular,/home/heyang-li/research/MillenniumDB/evaluati...,1908485,3193392,1.67326,5101877
1,1,0.0,0.053860,0.11752,23.19810,0.28915,0.00008,33.0,1.0,23.369480,160.855469,1.0,True,icij-leak,ICIJ-Leak,naive,Q0,1,data-1,simple_data,simple_regular,/home/heyang-li/research/MillenniumDB/evaluati...,1908485,3193392,1.67326,5101877
2,2,0.0,0.043879,0.12455,20.01580,0.16305,0.00010,33.0,1.0,20.184229,160.937500,1.0,True,icij-leak,ICIJ-Leak,naive,Q0,1,data-1,simple_data,simple_regular,/home/heyang-li/research/MillenniumDB/evaluati...,1908485,3193392,1.67326,5101877


In [9]:
agg_leaf = (
    exec_df.groupby([
        "dataset", "dataset_key", "algorithm", "query_template", "variant",
        "data_complexity", "regular_complexity", "V", "E", "E_over_V", "V_plus_E"
    ], dropna=False)
    .agg(
        n_exec=("exec_idx", "count"),
        mean_total_ms=("total_ms", "mean"),
        median_total_ms=("total_ms", "median"),
        mean_z3_op_ms=("z3_operation_ms", "mean"),
        mean_z3_solver_ms=("z3_solver_ms", "mean"),
        mean_solver_memory_mb=("solver_memory_mb", "mean"),
        empty_ratio=("is_empty", lambda s: np.nan if s.isna().all() else s.fillna(False).mean()),
        mean_cardinality=("result_cardinality", "mean"),
        max_cardinality=("result_cardinality", "max"),
    )
    .reset_index()
)

topq = (
    exec_df.groupby(["dataset", "dataset_key", "algorithm", "query_template", "variant"])["total_ms"]
    .apply(top_quartile_stats)
    .reset_index(name="topq")
)
topq[["avg_top_25_runtime", "median_top_25_runtime"]] = pd.DataFrame(topq["topq"].tolist(), index=topq.index)
topq = topq.drop(columns=["topq"])

agg_leaf = agg_leaf.merge(topq, on=["dataset", "dataset_key", "algorithm", "query_template", "variant"], how="left")

print("Aggregated leaf rows:", len(agg_leaf))
agg_leaf.head(3)

Aggregated leaf rows: 864


,dataset,dataset_key,algorithm,query_template,variant,data_complexity,regular_complexity,V,E,E_over_V,V_plus_E,n_exec,mean_total_ms,median_total_ms,mean_z3_op_ms,mean_z3_solver_ms,mean_solver_memory_mb,empty_ratio,mean_cardinality,max_cardinality,avg_top_25_runtime,median_top_25_runtime
0,ICIJ-Leak,icij-leak,naive,1,data-1,simple_data,simple_regular,1908485,3193392,1.67326,5101877,100,4.041993,0.228290,0.074183,0.000148,121.252525,1.0,0.95,1.0,15.227640,14.86536
1,ICIJ-Leak,icij-leak,naive,1,data-2,simple_data,simple_regular,1908485,3193392,1.67326,5101877,100,0.270239,0.203145,0.057225,0.000135,121.616162,1.0,0.95,1.0,0.405011,0.38247
2,ICIJ-Leak,icij-leak,naive,1,data-3,complex_data,simple_regular,1908485,3193392,1.67326,5101877,100,0.249490,0.187055,0.057523,0.000122,119.414141,1.0,0.95,1.0,0.394906,0.36470


In [10]:
# Runtime distributions by dataset and algorithm
fig, ax = plt.subplots(figsize=(14, 7))
sns.boxplot(data=exec_df, x="dataset", y="total_ms", hue="algorithm", ax=ax, showfliers=False)
ax.set_title("Runtime Distribution by Dataset (Naive vs Optimized)")
ax.set_xlabel("Dataset")
ax.set_ylabel("Total Runtime (ms)")
ax.tick_params(axis="x", rotation=25)
savefig_pdf(FIGURE_ROOT / "runtime" / "runtime_distribution_dataset_algo.pdf")

# Runtime by data complexity
fig, ax = plt.subplots(figsize=(12, 7))
sub = exec_df[exec_df["data_complexity"] != "no_data"].copy()
sns.violinplot(data=sub, x="data_complexity", y="total_ms", hue="algorithm", split=False, inner="quart", ax=ax)
ax.set_title("Runtime by Data Complexity")
ax.set_xlabel("Data Complexity")
ax.set_ylabel("Total Runtime (ms)")
savefig_pdf(FIGURE_ROOT / "runtime" / "runtime_by_data_complexity.pdf")

# Runtime by regular complexity
fig, ax = plt.subplots(figsize=(12, 7))
sns.violinplot(data=exec_df, x="regular_complexity", y="total_ms", hue="algorithm", split=False, inner="quart", ax=ax)
ax.set_title("Runtime by Regular Query Complexity")
ax.set_xlabel("Regular Complexity")
ax.set_ylabel("Total Runtime (ms)")
savefig_pdf(FIGURE_ROOT / "runtime" / "runtime_by_regular_complexity.pdf")

# Average top 25 runtime by category
cat_summary = (
    exec_df.groupby(["data_complexity", "algorithm"]) ["total_ms"]
    .apply(lambda s: top_quartile_stats(s)[0])
    .reset_index(name="avg_top_25_runtime")
)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=cat_summary, x="data_complexity", y="avg_top_25_runtime", hue="algorithm", ax=ax)
ax.set_title("Average Top 25% Runtime by Data Complexity")
ax.set_ylabel("Average Top 25% Runtime (ms)")
savefig_pdf(FIGURE_ROOT / "runtime" / "avg_top25_by_data_complexity.pdf")

print("Runtime figures exported")

Runtime figures exported


In [11]:
# Cardinality distributions by dataset
fig, ax = plt.subplots(figsize=(14, 7))
sns.boxplot(data=exec_df, x="dataset", y="result_cardinality", hue="algorithm", showfliers=False, ax=ax)
ax.set_title("Result Cardinality Distribution by Dataset")
ax.set_xlabel("Dataset")
ax.set_ylabel("Result Cardinality")
ax.tick_params(axis="x", rotation=25)
savefig_pdf(FIGURE_ROOT / "cardinality" / "cardinality_distribution_dataset_algo.pdf")

# Scatter: cardinality distribution per dataset with zero baseline
fig, ax = plt.subplots(figsize=(14, 7))
plot_df = exec_df.dropna(subset=["result_cardinality"]).copy()
for i, ds in enumerate(sorted(plot_df["dataset"].unique())):
    sub = plot_df[plot_df["dataset"] == ds]
    x = np.random.normal(i, 0.08, size=len(sub))
    ax.scatter(x, sub["result_cardinality"], s=8, alpha=0.35, label=ds)
ax.axhline(0, color="red", linestyle="--", linewidth=1.8)
ax.set_xticks(range(len(sorted(plot_df["dataset"].unique()))))
ax.set_xticklabels(sorted(plot_df["dataset"].unique()), rotation=35)
ax.set_title("Cardinality Distribution Across Datasets")
ax.set_xlabel("Dataset")
ax.set_ylabel("Cardinality")
savefig_pdf(FIGURE_ROOT / "cardinality" / "cardinality_scatter_by_dataset.pdf")

# Ranking style summary (mean/median/max by query template)
rank = (
    exec_df.groupby(["dataset", "query_template"], dropna=False)
    .agg(mean_card=("result_cardinality", "mean"), median_card=("result_cardinality", "median"), max_card=("result_cardinality", "max"))
    .reset_index()
)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
example_ds = "ICIJ-Leak"
r = rank[rank["dataset"] == example_ds].sort_values("mean_card", ascending=False)
axes[0, 0].bar([f"Q{int(x)}" for x in r["query_template"]], r["mean_card"], color="skyblue")
axes[0, 0].set_title(f"Mean Cardinality Ranking - {example_ds}")
axes[0, 0].tick_params(axis="x", rotation=45)

axes[0, 1].scatter(r["mean_card"], r["median_card"], color="steelblue")
if len(r) > 1:
    mn, mx = float(r["mean_card"].min()), float(r["mean_card"].max())
    axes[0, 1].plot([mn, mx], [mn, mx], "r--", linewidth=1)
axes[0, 1].set_title("Mean vs Median Cardinality")
axes[0, 1].set_xlabel("Mean")
axes[0, 1].set_ylabel("Median")

corr = r[["mean_card", "median_card", "max_card"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Reds", cbar=True, ax=axes[1, 0])
axes[1, 0].set_title("Ranking Correlation Matrix")

axes[1, 1].bar([f"Q{int(x)}" for x in r["query_template"]], r["max_card"], color="lightcoral")
axes[1, 1].set_title("Maximum Cardinality by Query")
axes[1, 1].tick_params(axis="x", rotation=45)

savefig_pdf(FIGURE_ROOT / "cardinality" / "ranking_panels_icij_leak.pdf")

print("Cardinality figures exported")

Cardinality figures exported


In [12]:
# Correlation matrices for overall and per complexity category
corr_features = ["V_plus_E", "E_over_V", "V", "E", "avg_top_25_runtime", "median_top_25_runtime"]

corr_base = agg_leaf[["data_complexity", *corr_features]].copy()

for name, sub in {
    "combined": corr_base,
    "simple_data": corr_base[corr_base["data_complexity"] == "simple_data"],
    "complex_data": corr_base[corr_base["data_complexity"] == "complex_data"],
}.items():
    m = sub[corr_features].corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(m, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=ax)
    ax.set_title(f"Correlation Matrix - {name}")
    savefig_pdf(FIGURE_ROOT / "correlation" / f"correlation_matrix_{name}.pdf")

# Runtime vs graph metrics (top-25 runtime summary)
plot_sum = (
    agg_leaf.groupby(["dataset", "dataset_key", "V", "E", "E_over_V", "V_plus_E"], as_index=False)
    .agg(avg_top_25_runtime=("avg_top_25_runtime", "mean"), median_top_25_runtime=("median_top_25_runtime", "mean"))
)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for ds, g in plot_sum.groupby("dataset"):
    axes[0, 0].scatter(g["V_plus_E"], g["avg_top_25_runtime"], label=ds, s=45, alpha=0.8)
    axes[0, 1].scatter(g["E_over_V"], g["avg_top_25_runtime"], label=ds, s=45, alpha=0.8)
    axes[1, 0].scatter(g["V"], g["avg_top_25_runtime"], label=ds, s=45, alpha=0.8)
    axes[1, 1].scatter(g["E"], g["avg_top_25_runtime"], label=ds, s=45, alpha=0.8)

axes[0, 0].set_title("Runtime vs |V| + |E|")
axes[0, 0].set_xlabel("|V| + |E|")
axes[0, 0].set_ylabel("Average Top 25% Runtime (ms)")

axes[0, 1].set_title("Runtime vs |E| / |V|")
axes[0, 1].set_xlabel("|E| / |V|")
axes[0, 1].set_ylabel("Average Top 25% Runtime (ms)")

axes[1, 0].set_title("Runtime vs |V|")
axes[1, 0].set_xlabel("|V|")
axes[1, 0].set_ylabel("Average Top 25% Runtime (ms)")

axes[1, 1].set_title("Runtime vs |E|")
axes[1, 1].set_xlabel("|E|")
axes[1, 1].set_ylabel("Average Top 25% Runtime (ms)")

handles, labels = axes[0, 0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="upper right")

savefig_pdf(FIGURE_ROOT / "correlation" / "runtime_vs_graph_metrics.pdf")
print("Correlation figures exported")

Correlation figures exported


In [13]:
# Z3 comparison: naive vs optimized split by empty/non-empty
z3_df = exec_df.dropna(subset=["is_empty"]).copy()
z3_df["empty_group"] = np.where(z3_df["is_empty"], "empty", "non_empty")

z3_summary = (
    z3_df.groupby(["dataset", "empty_group", "algorithm"], as_index=False)
    .agg(
        n=("exec_idx", "count"),
        mean_z3_operation_ms=("z3_operation_ms", "mean"),
        median_z3_operation_ms=("z3_operation_ms", "median"),
        mean_z3_solver_ms=("z3_solver_ms", "mean"),
        median_z3_solver_ms=("z3_solver_ms", "median"),
        mean_total_ms=("total_ms", "mean"),
    )
)

pivot_op = z3_summary.pivot_table(index=["dataset", "empty_group"], columns="algorithm", values="mean_z3_operation_ms")
pivot_solver = z3_summary.pivot_table(index=["dataset", "empty_group"], columns="algorithm", values="mean_z3_solver_ms")

for pvt, col in [(pivot_op, "z3_operation_speedup"), (pivot_solver, "z3_solver_speedup")]:
    if {"naive", "optimized"}.issubset(set(pvt.columns)):
        pvt[col] = pvt["naive"] / pvt["optimized"].replace({0: np.nan})

speedup = pivot_op[["z3_operation_speedup"]].join(pivot_solver[["z3_solver_speedup"]], how="outer").reset_index()

# Statistical tests and effect sizes
stat_rows = []
for (dataset, empty_group), sub in z3_df.groupby(["dataset", "empty_group"]):
    nsub = sub[sub["algorithm"] == "naive"]
    osub = sub[sub["algorithm"] == "optimized"]

    x1 = nsub["z3_operation_ms"].dropna().to_numpy()
    y1 = osub["z3_operation_ms"].dropna().to_numpy()
    x2 = nsub["z3_solver_ms"].dropna().to_numpy()
    y2 = osub["z3_solver_ms"].dropna().to_numpy()

    stat_rows.append({
        "dataset": dataset,
        "empty_group": empty_group,
        "n_naive": len(x1),
        "n_optimized": len(y1),
        "op_pvalue_mwu": safe_mwu_pvalue(x1, y1),
        "op_cliffs_delta": effect_size_cliffs_delta(x1, y1),
        "solver_pvalue_mwu": safe_mwu_pvalue(x2, y2),
        "solver_cliffs_delta": effect_size_cliffs_delta(x2, y2),
    })

z3_stats = pd.DataFrame(stat_rows)

# Export summary tables
z3_summary.to_csv(FIGURE_ROOT / "tables" / "z3_summary_by_dataset_empty_algorithm.csv", index=False)
speedup.to_csv(FIGURE_ROOT / "tables" / "z3_speedup_empty_nonempty.csv", index=False)
z3_stats.to_csv(FIGURE_ROOT / "tables" / "z3_stats_mwu_cliffs_delta.csv", index=False)

z3_summary.head(8)

,dataset,empty_group,algorithm,n,mean_z3_operation_ms,median_z3_operation_ms,mean_z3_solver_ms,median_z3_solver_ms,mean_total_ms
0,ICIJ-Leak,empty,naive,6806,0.067144,0.000020,0.000383,0.000100,1.186800
1,ICIJ-Leak,empty,optimized,6895,0.109270,0.000000,0.162914,0.000180,1.483419
2,ICIJ-Leak,non_empty,naive,394,0.302712,0.278715,0.895204,0.872030,5.425133
3,ICIJ-Leak,non_empty,optimized,391,NaN,NaN,NaN,NaN,1.748741
4,ICIJ-Paradises,empty,naive,4845,0.109588,0.000020,0.555529,0.000150,1.500397
5,ICIJ-Paradises,empty,optimized,5147,0.139672,0.000000,0.180122,0.000180,0.724093
6,ICIJ-Paradises,non_empty,naive,2355,0.316912,0.298850,0.565094,0.530651,1.907234
7,ICIJ-Paradises,non_empty,optimized,2053,0.584345,0.532611,0.771470,0.647001,1.718304


In [13]:
# Z3 figures
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.boxplot(
    data=z3_df,
    x="empty_group",
    y="z3_operation_ms",
    hue="algorithm",
    showfliers=False,
    ax=axes[0],
)
axes[0].set_title("Z3 Operation Time: Empty vs Non-Empty")
axes[0].set_xlabel("Query Group")
axes[0].set_ylabel("Z3 Operation Time (ms)")

sns.boxplot(
    data=z3_df,
    x="empty_group",
    y="z3_solver_ms",
    hue="algorithm",
    showfliers=False,
    ax=axes[1],
)
axes[1].set_title("Z3 Solver Time: Empty vs Non-Empty")
axes[1].set_xlabel("Query Group")
axes[1].set_ylabel("Z3 Solver Time (ms)")

savefig_pdf(FIGURE_ROOT / "z3" / "z3_runtime_empty_vs_nonempty_naive_vs_optimized.pdf")

for metric, fname in [
    ("z3_operation_speedup", "z3_operation_speedup_heatmap.pdf"),
    ("z3_solver_speedup", "z3_solver_speedup_heatmap.pdf"),
]:
    mat = speedup.pivot(index="dataset", columns="empty_group", values=metric)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(mat, annot=True, fmt=".2f", cmap="RdYlGn", center=1.0, ax=ax)
    ax.set_title(f"{metric} (naive / optimized)")
    ax.set_xlabel("Group")
    ax.set_ylabel("Dataset")
    savefig_pdf(FIGURE_ROOT / "z3" / fname)

print("Z3 figures exported")

Z3 figures exported


## 6. Add Basic Validation Tests

In [14]:
assert not exec_df.empty, "Execution dataframe is empty"
assert (exec_df["algorithm"].isin(ALGORITHMS)).all(), "Unexpected algorithm labels"
assert exec_df["dataset"].nunique() >= 6, "Expected at least six datasets"

# Consistency check: if both metrics exist, worker_results and result_cardinality emptiness should mostly align.
consistency = exec_df.dropna(subset=["worker_results", "result_cardinality"]).copy()
if not consistency.empty:
    mismatch = ((consistency["worker_results"] == 0) != (consistency["result_cardinality"] == 0)).mean()
    print("emptiness mismatch ratio (db.log vs result.json):", round(float(mismatch), 4))
else:
    print("No overlap rows for db.log/result.json emptiness check")

coverage = (
    exec_df.groupby(["dataset", "algorithm"], as_index=False)
    .agg(rows=("exec_idx", "count"), leaves=("leaf_path", "nunique"))
)
coverage

emptiness mismatch ratio (db.log vs result.json): 0.2795


,dataset,algorithm,rows,leaves
0,ICIJ-Leak,naive,7200,72
1,ICIJ-Leak,optimized,7286,72
2,ICIJ-Paradises,naive,7200,72
3,ICIJ-Paradises,optimized,7200,72
4,LDBC01,naive,7200,72
5,LDBC01,optimized,7200,72
6,LDBC10,naive,7212,72
7,LDBC10,optimized,7212,72
8,Pokec,naive,7200,72
9,Pokec,optimized,7221,72


## 7. Inspect and Export Results

In [15]:
# Save core tables
agg_leaf.to_csv(FIGURE_ROOT / "tables" / "aggregated_leaf_metrics.csv", index=False)
exec_df.to_csv(FIGURE_ROOT / "tables" / "execution_level_metrics.csv", index=False)

# Build export manifest
pdf_files = sorted(FIGURE_ROOT.rglob("*.pdf"))
manifest = pd.DataFrame({"pdf": [str(p.relative_to(ROOT)) for p in pdf_files]})
manifest.to_csv(FIGURE_ROOT / "tables" / "pdf_manifest.csv", index=False)

print(f"Exported {len(pdf_files)} PDF figures to: {FIGURE_ROOT}")
manifest

Exported 14 PDF figures to: /home/heyang-li/research/MillenniumDB/evaluation/figure/case-study-summary


,pdf
0,figure/case-study-summary/cardinality/cardinal...
1,figure/case-study-summary/cardinality/cardinal...
2,figure/case-study-summary/cardinality/ranking_...
3,figure/case-study-summary/correlation/correlat...
4,figure/case-study-summary/correlation/correlat...
5,figure/case-study-summary/correlation/correlat...
6,figure/case-study-summary/correlation/runtime_...
7,figure/case-study-summary/runtime/avg_top25_by...
8,figure/case-study-summary/runtime/runtime_by_d...
9,figure/case-study-summary/runtime/runtime_by_r...


In [16]:
# Focused summary for interpretation: where optimized helps most
merged_speed = speedup.copy()
merged_speed["op_helpful"] = merged_speed["z3_operation_speedup"] > 1.0
merged_speed["solver_helpful"] = merged_speed["z3_solver_speedup"] > 1.0

summary_interp = (
    merged_speed.groupby("empty_group", as_index=False)
    .agg(
        datasets_count=("dataset", "count"),
        op_median_speedup=("z3_operation_speedup", "median"),
        solver_median_speedup=("z3_solver_speedup", "median"),
        op_helpful_datasets=("op_helpful", "sum"),
        solver_helpful_datasets=("solver_helpful", "sum"),
    )
)
summary_interp

,empty_group,datasets_count,op_median_speedup,solver_median_speedup,op_helpful_datasets,solver_helpful_datasets
0,empty,6,4.394446,67.610159,4,5
1,non_empty,6,0.542337,0.732489,2,2


In [17]:

from IPython.display import Markdown, display
import pandas as pd
from pathlib import Path

# Resolve tables path the same way as the config cell
_root = Path.cwd()
if _root.name != "evaluation":
    _root = _root / "evaluation"
_tables = _root / "figure" / "case-study-summary" / "tables"

# Load from saved CSVs so this cell is self-contained
_speedup = pd.read_csv(_tables / "z3_speedup_empty_nonempty.csv")
_stats   = pd.read_csv(_tables / "z3_stats_mwu_cliffs_delta.csv")

# --- Z3 Speedup Table ---
sp = _speedup.copy()
sp.columns = ["Dataset", "Empty Group", "op_speedup (x)", "solver_speedup (x)"]
sp["op_speedup (x)"]     = sp["op_speedup (x)"].map(lambda x: f"{x:.2f}x" if pd.notna(x) else "-")
sp["solver_speedup (x)"] = sp["solver_speedup (x)"].map(lambda x: f"{x:.2f}x" if pd.notna(x) else "-")

# --- Z3 Stats Table ---
st = _stats[["dataset","empty_group","n_naive","n_optimized",
             "op_pvalue_mwu","op_cliffs_delta",
             "solver_pvalue_mwu","solver_cliffs_delta"]].copy()
st.columns = ["Dataset","Empty Group","n (naive)","n (opt)",
              "op p-val (MWU)","op Cliff delta",
              "solver p-val (MWU)","solver Cliff delta"]
for col in ["op p-val (MWU)", "solver p-val (MWU)"]:
    st[col] = st[col].map(lambda x: f"{x:.2e}" if pd.notna(x) else "-")
for col in ["op Cliff delta", "solver Cliff delta"]:
    st[col] = st[col].map(lambda x: f"{x:+.3f}" if pd.notna(x) else "-")

lines = [
    "## Z3 Speedup: Optimized vs Naive\n\n",
    "Speedup = mean(naive) / mean(optimized). Values > 1 mean optimized is faster.\n\n",
    sp.to_markdown(index=False),
    "\n\n---\n\n",
    "## Z3 Statistical Tests: Mann-Whitney U + Cliff's Delta\n\n",
    "Cliff delta: +1 = naive always larger (optimized faster). |d| > 0.47 = large effect.\n\n",
    st.to_markdown(index=False),
]
display(Markdown("".join(lines)))


## Z3 Speedup: Optimized vs Naive

Speedup = mean(naive) / mean(optimized). Values > 1 mean optimized is faster.

| Dataset        | Empty Group   | op_speedup (x)   | solver_speedup (x)   |
|:---------------|:--------------|:-----------------|:---------------------|
| ICIJ-Leak      | empty         | 0.61x            | 0.00x                |
| ICIJ-Leak      | non_empty     | -                | -                    |
| ICIJ-Paradises | empty         | 0.78x            | 3.08x                |
| ICIJ-Paradises | non_empty     | 0.54x            | 0.73x                |
| LDBC01         | empty         | 6.57x            | 122.60x              |
| LDBC01         | non_empty     | 26.64x           | 255.53x              |
| LDBC10         | empty         | 67.39x           | 1025.94x             |
| LDBC10         | non_empty     | 0.09x            | 0.06x                |
| Pokec          | empty         | 6.42x            | 111.07x              |
| Pokec          | non_empty     | 11.94x           | 36.24x               |
| Telecom        | empty         | 2.37x            | 24.15x               |
| Telecom        | non_empty     | 0.13x            | 0.72x                |

---

## Z3 Statistical Tests: Mann-Whitney U + Cliff's Delta

Cliff delta: +1 = naive always larger (optimized faster). |d| > 0.47 = large effect.

| Dataset        | Empty Group   |   n (naive) |   n (opt) | op p-val (MWU)   | op Cliff delta   | solver p-val (MWU)   | solver Cliff delta   |
|:---------------|:--------------|------------:|----------:|:-----------------|:-----------------|:---------------------|:---------------------|
| ICIJ-Leak      | empty         |        5932 |      6026 | 8.12e-93         | +0.211           | 0.00e+00             | -0.415               |
| ICIJ-Leak      | non_empty     |           8 |         0 | -                | -                | -                    | -                    |
| ICIJ-Paradises | empty         |        4367 |      4651 | 0.00e+00         | +0.460           | 1.23e-08             | -0.069               |
| ICIJ-Paradises | non_empty     |        1573 |      1289 | 0.00e+00         | -0.938           | 4.86e-231            | -0.704               |
| LDBC01         | empty         |        5915 |      5934 | 0.00e+00         | +0.995           | 9.87e-142            | -0.267               |
| LDBC01         | non_empty     |          25 |         6 | 1.05e-01         | -0.440           | 6.43e-01             | -0.133               |
| LDBC10         | empty         |        3571 |      3731 | 9.31e-292        | +0.493           | 0.00e+00             | +0.598               |
| LDBC10         | non_empty     |        2369 |      2209 | 0.00e+00         | -0.775           | 0.00e+00             | -0.701               |
| Pokec          | empty         |        1851 |      2693 | 5.88e-301        | -0.646           | 3.36e-281            | -0.624               |
| Pokec          | non_empty     |        4089 |      3267 | 0.00e+00         | -0.616           | 9.43e-134            | -0.333               |
| Telecom        | empty         |        4053 |      4694 | 0.00e+00         | +0.573           | 1.56e-38             | -0.159               |
| Telecom        | non_empty     |        1887 |      1385 | 5.19e-68         | -0.356           | 3.65e-54             | -0.317               |

Dataset	Empty queries	Non-empty queries
LDBC10	67.4×	0.09× (10× slower!)
Telecom	7.2×	0.42× (slower)
LDBC01	6.6×	26.6× (but n=31, p=0.105 – not significant)
Pokec	6.4×	11.9×
ICIJ-Paradises	0.78×	0.54× (both slower)
ICIJ-Leak	0.61×	N/A (0 optimized non-empty rows)


Z3 solver_time speedup:

Dataset	Empty queries	Non-empty queries
LDBC10	1026×	0.06× (16× slower!)
LDBC01	123×	256× (not significant)
Pokec	111×	36×
Telecom	95×	3.5×
ICIJ-Paradises	3.1×	0.73×
ICIJ-Leak	0.002×	N/A

Interpretation guide:

- `speedup > 1` means optimized is faster than naive for that metric.
- Compare `empty` vs `non_empty` rows in the final table to see where optimization gains concentrate.
- Use `z3_stats_mwu_cliffs_delta.csv` for significance/effect-size reporting in writeups.